# 📝 Text Classification: RNN vs. Transformer

**CO5085 – Assignment 1**

So sánh: **BiLSTM, GRU** vs. **DistilBERT**

In [ ]:
import sys
sys.path.insert(0, '../src')
import torch, os
os.makedirs('../results', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Data (20 Newsgroups + DistilBERT Tokenizer)

In [ ]:
from datasets import get_20newsgroups_loaders
train_loader, val_loader, test_loader, tokenizer, num_classes = get_20newsgroups_loaders(
    tokenizer_name='distilbert-base-uncased', batch_size=32, max_length=256)
print(f'Classes: {num_classes} | Train batches: {len(train_loader)}')

## 2. Model 1: DistilBERT (Transformer)

In [ ]:
from models import get_distilbert
from train import train

distilbert = get_distilbert(num_classes=num_classes)
history_distilbert = train(distilbert, train_loader, val_loader,
                           num_epochs=5, lr=2e-5, weight_decay=1e-2,
                           device=DEVICE, save_path='../results/distilbert_best.pt',
                           scheduler_type='cosine')

## 3. BiLSTM (RNN)

> **Note**: BiLSTM uses a simple vocabulary tokenizer instead of WordPiece.

In [ ]:
# Build vocab from training data
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch
import numpy as np

raw = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
from collections import Counter
words = Counter(' '.join(raw.data).lower().split())
vocab = {w: i+2 for i,(w,_) in enumerate(words.most_common(50000))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1
VOCAB_SIZE = len(vocab)

def tokenize_simple(texts, max_len=256):
    result = []
    for text in texts:
        ids = [vocab.get(w, 1) for w in text.lower().split()[:max_len]]
        ids = ids + [0] * (max_len - len(ids))
        result.append(ids)
    return torch.tensor(result, dtype=torch.long)

tr_texts, val_texts, tr_labels, val_labels = train_test_split(
    raw.data, raw.target, test_size=0.15, random_state=42)

tr_X  = tokenize_simple(tr_texts)
val_X = tokenize_simple(val_texts)
test_raw = fetch_20newsgroups(subset='test', remove=('headers','footers','quotes'))
test_X = tokenize_simple(test_raw.data)

train_lstm = DataLoader(TensorDataset(tr_X, torch.tensor(tr_labels, dtype=torch.long)), batch_size=64, shuffle=True)
val_lstm   = DataLoader(TensorDataset(val_X, torch.tensor(val_labels, dtype=torch.long)), batch_size=64)
test_lstm  = DataLoader(TensorDataset(test_X, torch.tensor(list(test_raw.target), dtype=torch.long)), batch_size=64)

print(f'Vocab size: {VOCAB_SIZE}')

In [ ]:
from models import BiLSTMClassifier
from train import train

bilstm = BiLSTMClassifier(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_dim=256,
                           num_layers=2, num_classes=20, dropout=0.4)
history_bilstm = train(bilstm, train_lstm, val_lstm,
                       num_epochs=15, lr=1e-3,
                       device=DEVICE, save_path='../results/bilstm_best.pt')

## 4. GRU (RNN)

In [ ]:
from models import GRUClassifier

gru = GRUClassifier(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_dim=256,
                    num_layers=2, num_classes=20, dropout=0.4)
history_gru = train(gru, train_lstm, val_lstm,
                    num_epochs=15, lr=1e-3,
                    device=DEVICE, save_path='../results/gru_best.pt')

## 5. Evaluation & Comparison

In [ ]:
from evaluate import compute_metrics, plot_training_curves, compare_models, get_predictions
import torch, numpy as np

results_text = {}

# DistilBERT
preds, labels, _ = get_predictions(distilbert, test_loader, DEVICE)
results_text['DistilBERT'] = compute_metrics(preds, labels, verbose=True)
plot_training_curves(history_distilbert, 'DistilBERT', '../results/distilbert_curves.png')

# BiLSTM & GRU
for rnn_name, rnn_model in [('BiLSTM', bilstm), ('GRU', gru)]:
    rnn_model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for xb, yb in test_lstm:
            out = rnn_model(xb.to(DEVICE))
            all_p += out.argmax(-1).cpu().tolist()
            all_l += yb.tolist()
    results_text[rnn_name] = compute_metrics(np.array(all_p), np.array(all_l), verbose=True)
    plot_training_curves(history_bilstm if rnn_name == 'BiLSTM' else history_gru,
                         rnn_name, f'../results/{rnn_name.lower()}_curves.png')

compare_models(results_text, metric='accuracy', save_path='../results/text_comparison_acc.png')
compare_models(results_text, metric='f1_macro',  save_path='../results/text_comparison_f1.png')